# 検索の評価セットを作って手法を比べる

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/25_rag/rag_eval_retrieval.ipynb)

NLP クックブック 25-10 ／ RAG（検索拡張生成）

RAG の検索部分を改善したい。BM25、埋め込み、ハイブリッドのどれが自分のデータに合うのかを、数値で比べたい。

In [1]:
!pip install -q rank_bm25 fugashi unidic-lite

In [2]:
import io, tarfile, urllib.request
import numpy as np
import pandas as pd

def load_livedoor():
    url = "https://www.rondhuit.com/download/ldcc-20140209.tar.gz"
    with urllib.request.urlopen(url) as r:
        tar = tarfile.open(fileobj=io.BytesIO(r.read()), mode="r:gz")
    rows = []
    for m in tar.getmembers():
        parts = m.name.split("/")
        if m.isfile() and len(parts) == 3 and parts[2] != "LICENSE.txt":
            lines = tar.extractfile(m).read().decode("utf-8").split("\n")
            rows.append({"label": parts[1], "title": lines[2], "text": "\n".join(lines[3:])})
    return pd.DataFrame(rows)

# 評価セット: 「記事のタイトル」を質問、「その記事の本文」を正解の文書とみなす (人手のラベルなしで作れる)
df = load_livedoor().sample(600, random_state=0).reset_index(drop=True)
corpus = [t[:400] for t in df["text"]]
queries = df["title"].tolist()[:200]            # 正解は同じ番号の文書
print(len(corpus), "文書", len(queries), "クエリ |", queries[0], "→", corpus[0][:40].replace("\n", " "))

600 文書 200 クエリ | 永遠の恋愛テーマにハマった杉崎アナ、涌井投手が破局 → 今年話題の恋愛・スキャンダルニュース＜7月＞  今年5月に報じられ本人からも交際


In [3]:
from fugashi import Tagger
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

tagger = Tagger()
tok = lambda s: [w.surface.lower() for w in tagger(s) if w.feature.pos1 not in ("助詞", "助動詞", "補助記号", "空白")]
bm25 = BM25Okapi([tok(d) for d in corpus])
model = SentenceTransformer("intfloat/multilingual-e5-small")
doc_emb = model.encode(["passage: " + d for d in corpus], normalize_embeddings=True, batch_size=32, show_progress_bar=False)
q_emb = model.encode(["query: " + q for q in queries], normalize_embeddings=True, batch_size=32, show_progress_bar=False)

sparse = np.array([np.argsort(-bm25.get_scores(tok(q))) for q in queries])      # 各クエリについて、文書番号を良い順に並べたもの
dense = np.argsort(-(q_emb @ doc_emb.T), axis=1)

def rrf(rankings, k=60):
    scores = np.zeros(rankings[0].shape)
    for r in rankings:
        scores[np.arange(len(r))[:, None], r] += 1 / (k + np.arange(1, r.shape[1] + 1))
    return np.argsort(-scores, axis=1)

def evaluate(ranking):
    rank = np.array([np.where(ranking[i] == i)[0][0] + 1 for i in range(len(ranking))])   # 正解文書の順位
    return {"Recall@1": (rank <= 1).mean(), "Recall@5": (rank <= 5).mean(), "Recall@20": (rank <= 20).mean(), "MRR": (1 / rank).mean()}

pd.DataFrame({"BM25": evaluate(sparse), "埋め込み": evaluate(dense), "ハイブリッド (RRF)": evaluate(rrf([sparse, dense]))}).round(3).T

,Recall@1,Recall@5,Recall@20,MRR
BM25,0.895,0.975,0.990,0.932
埋め込み,0.910,0.985,0.990,0.939
ハイブリッド (RRF),0.930,0.985,0.995,0.955


In [4]:
# どちらか一方しか当てられなかったクエリを見ると、手法の得意・不得意が分かる
rank_s = np.array([np.where(sparse[i] == i)[0][0] for i in range(len(queries))])
rank_d = np.array([np.where(dense[i] == i)[0][0] for i in range(len(queries))])
print("BM25 だけ 1 位:", [queries[i] for i in np.where((rank_s == 0) & (rank_d > 4))[0][:3]])
print("埋め込みだけ 1 位:", [queries[i] for i in np.where((rank_d == 0) & (rank_s > 4))[0][:3]])

BM25 だけ 1 位: []
埋め込みだけ 1 位: ['女性が抱える“アノ悩み”は、身近な食物と食べ方に解決の光明あり']


- RAG の改善は、まず検索の評価セットを作るところから。50〜200 件の「質問と正解文書の組」があれば、手法やチャンク設定の比較ができる。
- この例では、Recall@1 も MRR も埋め込み（0.910、0.939）が BM25（0.895、0.932）をわずかに上回り、ハイブリッド（0.930、0.955）が最も高い。最後のセルでは、BM25 が 1 位で埋め込みが 6 位以下のクエリは 1 件もなく（空リスト）、逆は 1 件だけだった。差がつくクエリが少ないときは、評価セットを増やすか、質問を難しくして比べる。
- タイトルを質問に使う方法は手軽だが、実際のユーザーの質問とは性質が違う（本文と同じ語を含みやすく、BM25 に有利）。実運用のログや、LLM に文書から質問を作らせた合成データも併用する。
- Recall@k の k は、LLM に渡す件数に合わせる。5 件渡すなら Recall@5 が、そのまま「正解を読ませられた割合」になる。